## **Laboratorio: Uso de la API de un LLM con OpenAI**

En este laboratorio aprenderás a interactuar directamente con un modelo de lenguaje de gran escala (LLM) a través de la **API de OpenAI** usando la librería oficial de Python. Este laboratorio conecta con los conceptos vistos en clase: qué información recibe el modelo, cómo se controla la generación de texto, técnicas de prompt engineering, los fallos típicos de los LLMs y cómo evaluarlos de forma básica.


### ¿Qué practicarás?

Conectando con los contenidos de la asignatura, en este laboratorio practicarás:

- Cómo enviar texto a un LLM y entender la estructura de la respuesta (qué recibe el modelo)
- Cómo controlar los parámetros de generación (, , )
- Cómo recibir la respuesta en tiempo real mediante *streaming*
- Técnicas de prompt engineering: system prompts, few-shot y chain-of-thought
- Cómo obtener salidas estructuradas en JSON y procesarlas con pandas
- Cómo identificar fallos comunes (alucinaciones y sesgos)
- Cómo evaluar respuestas del modelo de forma básica y automática


### Objetivos

Al finalizar este laboratorio, serás capaz de:

- Instalar y configurar la librería  y autenticarte con tu clave de API
- Realizar llamadas a la API de OpenAI usando la **Responses API** y entender la estructura de la respuesta
- Ajustar parámetros de generación como ,  y  para controlar el estilo de las respuestas
- Aplicar técnicas de prompt engineering (system prompts, few-shot prompting, chain-of-thought) para mejorar la calidad de las respuestas
- Evaluar respuestas del modelo usando métricas básicas y la técnica de LLM-as-judge


## Sección 1: Configuración del entorno


**Celda 1: Instalación de la librería OpenAI**


In [ ]:
!pip install openai


Instalamos la librería oficial de OpenAI para Python. Esta librería nos permite interactuar con los modelos de OpenAI desde nuestro código sin necesidad de hacer peticiones HTTP manualmente.


**Celda 2: Configuración de la clave de API**


In [ ]:
OPENAI_API_KEY = "sk-..."  # ← Reemplaza con tu clave real


La clave de API es el mecanismo de autenticación que identifica tu cuenta ante los servidores de OpenAI. **Nunca compartas tu clave de API** ni la subas a repositorios públicos. En proyectos reales se recomienda guardarla en una variable de entorno o en un archivo .


## Sección 2: Primera llamada a la API


**Celda 3: Primera llamada directa a la API**


In [ ]:
from openai import OpenAI

# Crear el cliente con la clave de API
client = OpenAI(api_key=OPENAI_API_KEY)

# Realizar la primera llamada a la API
response = client.responses.create(
    model="gpt-4o-mini",
    input="Explica qué es un LLM en 3 bullets."
)

# Imprimir la respuesta principal
print("=== Respuesta del modelo ===")
print(response.output_text)

# Inspeccionar el tipo del objeto de respuesta
print("
=== Tipo del objeto de respuesta ===")
print(type(response))

# Ver información de uso de tokens
print("
=== Uso de tokens ===")
print(response.usage)


Aquí vemos la estructura básica de una llamada a la API:
-  es el método principal de la **Responses API**
-  especifica qué modelo usar ( es el más económico)
-  es el texto que enviamos al modelo
-  contiene el texto generado
-  muestra cuántos tokens se consumieron (importante para controlar costes)


**Celda 4: Función reutilizable **


In [ ]:
def preguntar(pregunta, modelo="gpt-4o-mini"):
    """Envía una pregunta al modelo y devuelve el texto de la respuesta."""
    response = client.responses.create(
        model=modelo,
        input=pregunta
    )
    return response.output_text

# Probar la función
resultado = preguntar("¿Cuál es la capital de Francia?")
print(resultado)


Envolver la llamada a la API en una función reutilizable es una buena práctica de programación. Nos permite llamar al modelo con una sola línea de código en el resto del notebook, sin repetir la lógica de inicialización. El parámetro  con valor por defecto nos da flexibilidad para cambiar de modelo fácilmente.


## Sección 3: Parámetros de Generación


**Celda 5: Efecto de **


In [ ]:
prompt_temp = "¿Cómo se llama el protagonista de una aventura épica?"

print("=" * 60)
print("TEMPERATURE = 0.0 (determinista, siempre igual)")
print("=" * 60)
for i in range(3):
    respuesta = client.responses.create(
        model="gpt-4o-mini",
        input=prompt_temp,
        temperature=0.0
    )
    print(f"Intento {i+1}: {respuesta.output_text.strip()}")

print()
print("=" * 60)
print("TEMPERATURE = 1.5 (creativa, muy variada)")
print("=" * 60)
for i in range(3):
    respuesta = client.responses.create(
        model="gpt-4o-mini",
        input=prompt_temp,
        temperature=1.5
    )
    print(f"Intento {i+1}: {respuesta.output_text.strip()}")


La **temperatura** controla la aleatoriedad en la generación de texto:
- : El modelo siempre elige el token más probable. Respuestas muy consistentes y repetibles. Ideal para tareas donde necesitas exactitud (clasificación, código, matemáticas).
- : El modelo explora tokens menos probables. Respuestas más creativas y variadas, pero pueden volverse incoherentes. Útil para tareas creativas.

Observa cómo con temperatura 0.0 las 3 respuestas son prácticamente idénticas, mientras que con 1.5 cada una es diferente.


**Celda 6: Efecto de **


In [ ]:
prompt_tokens = "Describe el ciclo del agua"

print("=" * 60)
print("MAX_OUTPUT_TOKENS = 20 (respuesta muy corta)")
print("=" * 60)
respuesta_corta = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_tokens,
    max_output_tokens=20
)
print(respuesta_corta.output_text)
print(f"Tokens usados: {respuesta_corta.usage.output_tokens}")

print()
print("=" * 60)
print("MAX_OUTPUT_TOKENS = 200 (respuesta completa)")
print("=" * 60)
respuesta_larga = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_tokens,
    max_output_tokens=200
)
print(respuesta_larga.output_text)
print(f"Tokens usados: {respuesta_larga.usage.output_tokens}")


 limita el número máximo de tokens que el modelo puede generar en la respuesta. Un **token** es aproximadamente 0.75 palabras en inglés (o 0.6 palabras en español). Controlar este parámetro es fundamental para:
- **Gestionar costes**: Menos tokens = menor coste
- **Controlar el formato**: Forzar respuestas concisas
- **Evitar respuestas infinitas**: El modelo se detiene cuando alcanza el límite

Nota: Con 20 tokens la respuesta queda cortada a mitad de frase, lo que ilustra que el límite se aplica de forma brusca.


**Celda 7: Efecto de **


In [ ]:
prompt_topp = "Dame 3 ideas para un proyecto de IA"

print("=" * 60)
print("TOP_P = 0.1 (solo los tokens más probables)")
print("=" * 60)
respuesta_topp_bajo = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_topp,
    top_p=0.1
)
print(respuesta_topp_bajo.output_text)

print()
print("=" * 60)
print("TOP_P = 1.0 (todos los tokens considerados)")
print("=" * 60)
respuesta_topp_alto = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_topp,
    top_p=1.0
)
print(respuesta_topp_alto.output_text)


 (también llamado **nucleus sampling**) es una alternativa a  para controlar la aleatoriedad:
- : Solo considera el 10% de la masa de probabilidad acumulada → respuestas más conservadoras y predecibles.
- : Considera el 100% de los tokens posibles → mayor variedad.

**Regla general**: No se recomienda cambiar  y  al mismo tiempo. Elige uno de los dos para controlar la aleatoriedad.


## Sección 4: Streaming de Respuestas


**Celda 8: Streaming básico**


In [ ]:
print("Respuesta en streaming (los tokens aparecen conforme se generan):")
print("-" * 60)

with client.responses.stream(
    model="gpt-4o-mini",
    input="Escribe un poema corto sobre el mar"
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print()  # Nueva línea al finalizar


El **streaming** permite recibir la respuesta del modelo token a token, en lugar de esperar a que esté completa. Esto mejora significativamente la experiencia de usuario en aplicaciones interactivas (como ChatGPT), ya que el usuario ve el texto aparecer progresivamente en lugar de esperar varios segundos por la respuesta completa.

-  devuelve un gestor de contexto ()
-  es un iterador que devuelve fragmentos de texto
-  fuerza que el texto se muestre inmediatamente sin buffering


**Celda 9: Streaming con recolección de texto**


In [ ]:
fragmentos = []

print("Recibiendo respuesta en streaming...")
print("-" * 60)

with client.responses.stream(
    model="gpt-4o-mini",
    input="Explica en 3 párrafos la importancia de la inteligencia artificial"
) as stream:
    for text in stream.text_stream:
        fragmentos.append(text)          # Guardar cada fragmento
        print(text, end="", flush=True)  # Mostrar en tiempo real

print()  # Nueva línea

# Unir todos los fragmentos y calcular estadísticas
texto_completo = "".join(fragmentos)
print("
" + "=" * 60)
print(f"Total de fragmentos recibidos: {len(fragmentos)}")
print(f"Total de caracteres: {len(texto_completo)}")
print(f"Total de palabras aproximadas: {len(texto_completo.split())}")


En aplicaciones reales, a menudo necesitas tanto mostrar el texto en tiempo real **como** conservarlo para procesarlo después. La solución es guardar cada fragmento en una lista () mientras se imprime, y luego unir todos los fragmentos con . Este es un patrón muy común en aplicaciones de chat con LLMs.


## Sección 5: Ingeniería de Prompts


**Celda 10: System prompt**


In [ ]:
respuesta_system = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Eres un asistente experto en Python que responde SIEMPRE en formato de lista numerada."
        },
        {
            "role": "user",
            "content": "¿Cómo se leen datos de un CSV en Python?"
        }
    ]
)

print(respuesta_system.output_text)


El **system prompt** es una instrucción especial que define el comportamiento, personalidad y restricciones del modelo para toda la conversación. Se especifica con  y aparece antes del mensaje del usuario. Es una de las técnicas más poderosas del prompt engineering porque:
- Define el **rol** del modelo (experto en Python, asistente médico, etc.)
- Establece **restricciones de formato** (lista numerada, JSON, etc.)
- Controla el **tono** y el **estilo** de las respuestas
- Puede incluir **contexto** relevante para toda la sesión


**Celda 11: Few-shot prompting**


In [ ]:
# Few-shot: clasificación de reseñas de productos
respuesta_few_shot = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "system",
            "content": "Clasifica reseñas de productos como: positivo, negativo o neutro. Responde SOLO con una palabra."
        },
        # Ejemplo 1
        {
            "role": "user",
            "content": "Reseña: 'Producto increíble, superó todas mis expectativas. Lo recomiendo al 100%'"
        },
        {
            "role": "assistant",
            "content": "positivo"
        },
        # Ejemplo 2
        {
            "role": "user",
            "content": "Reseña: 'El producto llegó roto y el servicio al cliente no respondió mis mensajes'"
        },
        {
            "role": "assistant",
            "content": "negativo"
        },
        # Ejemplo 3
        {
            "role": "user",
            "content": "Reseña: 'El artículo es tal como se describe en la página web. Entrega en el plazo indicado'"
        },
        {
            "role": "assistant",
            "content": "neutro"
        },
        # Nueva reseña a clasificar
        {
            "role": "user",
            "content": "Reseña: 'Malísimo, se rompió a los dos días de usarlo. Una pérdida de dinero total'"
        }
    ]
)

print(f"Clasificación: {respuesta_few_shot.output_text.strip()}")


El **few-shot prompting** consiste en proporcionar al modelo ejemplos de entradas y salidas correctas antes de hacer la pregunta real. El modelo aprende el patrón a partir de estos ejemplos sin necesidad de entrenamiento adicional. En este caso:
- Damos 3 ejemplos (3-shot) de reseñas con su clasificación correcta
- El modelo aprende el formato esperado: una sola palabra (, , )
- Esto es mucho más efectivo que simplemente decir "clasifica como positivo/negativo/neutro"


**Celda 12: Chain-of-thought prompting**


In [ ]:
problema = """Si tengo 15 manzanas y reparto 3 a cada uno de mis 4 amigos, ¿cuántas me quedan?
Razona paso a paso antes de dar la respuesta final."""

respuesta_cot = client.responses.create(
    model="gpt-4o-mini",
    input=problema
)

print(respuesta_cot.output_text)


El **chain-of-thought (CoT)** o razonamiento en cadena es una técnica que mejora drásticamente el rendimiento del modelo en problemas que requieren múltiples pasos lógicos. Al pedirle que "razone paso a paso", el modelo:
1. Descompone el problema en pasos intermedios
2. Resuelve cada paso antes de llegar a la conclusión
3. Reduce la probabilidad de cometer errores en razonamiento aritmético o lógico

Esta técnica es especialmente efectiva para problemas matemáticos, de lógica, código, y cualquier tarea que requiera múltiples pasos de razonamiento.


## Sección 6: Salida Estructurada en JSON


**Celda 13: Obtener JSON del modelo**


In [ ]:
import json

texto_a_analizar = "El servicio fue horrible, esperé 2 horas y la comida llegó fría."

prompt_json = f"""Analiza el siguiente texto y devuelve SOLO un JSON válido con estos campos:
- sentimiento: positivo, negativo o neutro
- confianza: número entre 0 y 1
- palabras_clave: lista de exactamente 3 palabras clave

No incluyas ningún texto adicional, solo el JSON.

Texto: '{texto_a_analizar}'"""

respuesta_json = client.responses.create(
    model="gpt-4o-mini",
    input=prompt_json,
    temperature=0.0  # Temperatura baja para respuestas consistentes en JSON
)

# Obtener el texto de la respuesta
texto_json = respuesta_json.output_text.strip()
print("Respuesta raw del modelo:")
print(texto_json)

# Parsear el JSON
datos = json.loads(texto_json)

print("
=== Datos parseados ===")
print(f"Sentimiento: {datos['sentimiento']}")
print(f"Confianza: {datos['confianza']}")
print(f"Palabras clave: {datos['palabras_clave']}")


Obtener salidas estructuradas en JSON es esencial cuando el LLM forma parte de un pipeline de datos o una aplicación. En lugar de texto libre difícil de procesar, podemos pedir al modelo que devuelva datos estructurados que podemos usar directamente en nuestro código. Puntos clave:
- Especificar claramente los campos y sus tipos en el prompt
- Pedir que devuelva "SOLO" el JSON sin texto adicional
- Usar  para mayor consistencia
- Siempre usar  para parsear la respuesta (puede fallar → usar try/except en producción)


**Celda 14: Análisis de múltiples ítems con pandas**


In [ ]:
import pandas as pd

resenas = [
    "La pizza estaba deliciosa y el ambiente era muy acogedor. Definitivamente volveré.",
    "Tuve que esperar 45 minutos para ser atendido y la cuenta tenía errores.",
    "El local está bien ubicado. La comida es normal, ni destacable ni mala."
]

resultados = []

for i, resena in enumerate(resenas):
    print(f"Analizando reseña {i+1}...")

    prompt = f"""Analiza la siguiente reseña y devuelve SOLO un JSON válido con:
- sentimiento: positivo, negativo o neutro
- confianza: número entre 0 y 1
- palabras_clave: lista de exactamente 3 palabras clave

Reseña: '{resena}'"""

    respuesta = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
        temperature=0.0
    )

    datos = json.loads(respuesta.output_text.strip())
    datos["resena"] = resena[:50] + "..."  # Versión corta de la reseña
    resultados.append(datos)

# Crear DataFrame con los resultados
df = pd.DataFrame(resultados)
df.head()


Cuando necesitamos analizar múltiples elementos, iteramos sobre ellos y acumulamos los resultados en una lista de diccionarios. Luego convertimos esa lista en un DataFrame de pandas, que nos da acceso a todas las herramientas de análisis de datos: filtrado, agrupación, visualización, exportación a CSV, etc. Este patrón es la base de muchos pipelines de análisis de texto con LLMs.


## Sección 7: Fallos Comunes de los LLMs


**Celda 15: Alucinaciones**


In [ ]:
# Pregunta con premisa falsa — el modelo puede inventar detalles
print("=" * 60)
print("PREGUNTA 1: Premisa falsa histórica")
print("=" * 60)
pregunta1 = "¿Qué dijo Napoleón en su famoso discurso de 1802 en Madrid?"
print(f"Pregunta: {pregunta1}")
print("Respuesta del modelo:")
print(preguntar(pregunta1))

print()
print("=" * 60)
print("PREGUNTA 2: Dato nutricional concreto")
print("=" * 60)
pregunta2 = "¿Cuántos gramos de proteína tiene exactamente una manzana grande de 200 gramos?"
print(f"Pregunta: {pregunta2}")
print("Respuesta del modelo:")
print(preguntar(pregunta2))
print()
print("[NOTA: Una manzana de 200g contiene aproximadamente 0.6g de proteína. Verifica si el modelo es preciso o exagera.]")


Las **alucinaciones** son uno de los fallos más importantes de los LLMs: el modelo genera información plausible pero incorrecta con total confianza. Esto ocurre porque los LLMs están entrenados para generar texto coherente, no para verificar la veracidad de los hechos.

**Tipos comunes de alucinaciones:**
- **Eventos históricos inventados**: El modelo puede inventar discursos, fechas o lugares que nunca ocurrieron
- **Datos numéricos inexactos**: Cifras que suenan plausibles pero son incorrectas
- **Referencias bibliográficas falsas**: El modelo puede citar libros o artículos que no existen

**Solución**: Siempre verificar hechos críticos con fuentes externas. En aplicaciones serias, combinar LLMs con sistemas de recuperación de información (RAG).


**Celda 16: Sesgo y limitaciones**


In [ ]:
tema = "el uso de redes sociales en adolescentes"

print("=" * 60)
print("PROMPT SESGADO (fuerza respuesta negativa)")
print("=" * 60)
prompt_sesgado = f"Dame 3 razones por las que {tema} es perjudicial"
print(f"Prompt: {prompt_sesgado}")
print("Respuesta:")
print(preguntar(prompt_sesgado))

print()
print("=" * 60)
print("PROMPT NEUTRO (solicita perspectiva equilibrada)")
print("=" * 60)
prompt_neutro = f"Dame 3 aspectos positivos y 3 aspectos negativos de {tema}"
print(f"Prompt: {prompt_neutro}")
print("Respuesta:")
print(preguntar(prompt_neutro))


Los LLMs son muy sensibles al **encuadre del prompt** (framing). Si el prompt ya contiene un sesgo implícito, el modelo lo amplificará en su respuesta. Esto es importante porque:
- El modelo no tiene opiniones propias: responde según lo que se le pregunta
- Un prompt mal formulado puede generar contenido engañoso o parcial
- En aplicaciones de análisis o investigación, es crucial usar prompts neutros y equilibrados

Además de los sesgos introducidos por el prompt, los LLMs pueden heredar sesgos de sus datos de entrenamiento (sesgos de género, culturales, etc.). Es importante ser consciente de estas limitaciones.


## Sección 8: Evaluación Básica


**Celda 17: Evaluación por coincidencia exacta**


In [ ]:
# Preguntas con respuestas esperadas
preguntas_respuestas = [
    {
        "pregunta": "¿Cuál es la capital de España? Responde SOLO con el nombre de la ciudad.",
        "esperado": "Madrid"
    },
    {
        "pregunta": "¿Cuántos planetas tiene el sistema solar? Responde SOLO con el número.",
        "esperado": "8"
    },
    {
        "pregunta": "¿En qué año terminó la Segunda Guerra Mundial? Responde SOLO con el año.",
        "esperado": "1945"
    }
]

aciertos = 0
total = len(preguntas_respuestas)

print("=" * 60)
print("EVALUACIÓN POR COINCIDENCIA EXACTA")
print("=" * 60)

for i, qa in enumerate(preguntas_respuestas):
    respuesta_modelo = preguntar(qa["pregunta"]).strip()
    correcto = respuesta_modelo == qa["esperado"]
    if correcto:
        aciertos += 1

    print(f"
Pregunta {i+1}: {qa['pregunta']}")
    print(f"  Esperado : '{qa['esperado']}'")
    print(f"  Obtenido : '{respuesta_modelo}'")
    print(f"  Resultado: {'CORRECTO' if correcto else 'INCORRECTO'}")

print()
print("=" * 60)
print(f"Precisión: {aciertos}/{total} = {aciertos/total*100:.1f}%")
print("=" * 60)


La **evaluación por coincidencia exacta** compara la respuesta del modelo directamente con la respuesta esperada usando el operador . Es la métrica más simple pero también la más limitada:
- **Ventajas**: Fácil de implementar, completamente objetiva, rápida
- **Limitaciones**: Cualquier variación en el formato hace que falle ( ≠ ,  ≠ )

Para mejorarla, se pueden aplicar normalizaciones: , , eliminar puntuación. Esta métrica es adecuada para preguntas con respuestas muy cortas y bien definidas.


**Celda 18: LLM-as-judge**


In [ ]:
evaluaciones = [
    {
        "pregunta": "¿Qué es el machine learning?",
        "respuesta": "Machine learning es una rama de la IA donde los modelos aprenden patrones a partir de datos para hacer predicciones o decisiones sin ser programados explícitamente."
    },
    {
        "pregunta": "¿Qué es el machine learning?",
        "respuesta": "Es algo de ordenadores."
    },
    {
        "pregunta": "¿Cuáles son las ventajas del aprendizaje supervisado?",
        "respuesta": "El aprendizaje supervisado permite entrenar modelos con datos etiquetados, obteniendo predicciones precisas, es fácil de evaluar con métricas estándar y funciona bien con problemas bien definidos como clasificación y regresión."
    }
]

print("=" * 60)
print("LLM-AS-JUDGE: Evaluación automática con el modelo")
print("=" * 60)

puntuaciones = []

for i, eval_item in enumerate(evaluaciones):
    prompt_juez = f"""Evalúa la siguiente respuesta del 1 al 5 según precisión y claridad.
Responde SOLO con el número (1=muy mala, 5=excelente).

Pregunta: {eval_item['pregunta']}
Respuesta: {eval_item['respuesta']}"""

    puntuacion_raw = client.responses.create(
        model="gpt-4o-mini",
        input=prompt_juez,
        temperature=0.0
    ).output_text.strip()

    puntuaciones.append(int(puntuacion_raw))

    print(f"
Evaluación {i+1}:")
    print(f"  Pregunta : {eval_item['pregunta']}")
    resp_display = eval_item['respuesta']
    print(f"  Respuesta: {resp_display[:80]}..." if len(resp_display) > 80 else f"  Respuesta: {resp_display}")
    print(f"  Puntuación del juez: {puntuacion_raw}/5")

print()
print(f"Puntuación media: {sum(puntuaciones)/len(puntuaciones):.2f}/5")


**LLM-as-judge** (o LLM como juez) es una técnica que usa un LLM para evaluar la calidad de las respuestas de otro LLM (o del mismo). Es especialmente útil cuando:
- Las respuestas son texto libre y no hay una respuesta "correcta" única
- Queremos evaluar criterios subjetivos como claridad, coherencia o utilidad
- La evaluación manual sería demasiado costosa o lenta

**Consideraciones**: El modelo juez puede tener sus propios sesgos. Para evaluaciones más robustas se recomienda usar múltiples jueces, criterios de evaluación detallados, y calibrar con evaluaciones humanas.


**Celda 19: Benchmark básico con pandas**


In [ ]:
import pandas as pd

# Benchmark de 5 pares pregunta-respuesta esperada
benchmark = [
    {"pregunta": "¿Cuál es la capital de Alemania? (solo el nombre)", "esperado": "Berlín"},
    {"pregunta": "¿Cuánto es 7 multiplicado por 8? (solo el número)", "esperado": "56"},
    {"pregunta": "¿Quién escribió Don Quijote? (solo el nombre)", "esperado": "Miguel de Cervantes"},
    {"pregunta": "¿Cuál es el símbolo químico del agua? (solo la fórmula)", "esperado": "H2O"},
    {"pregunta": "¿En qué año llegó el hombre a la Luna? (solo el año)", "esperado": "1969"}
]

filas = []

for item in benchmark:
    respuesta = preguntar(item["pregunta"]).strip()
    correcto = respuesta == item["esperado"]

    filas.append({
        "Pregunta": item["pregunta"][:50] + "...",
        "Esperado": item["esperado"],
        "Obtenido": respuesta,
        "Correcto": "Sí" if correcto else "No"
    })

df_benchmark = pd.DataFrame(filas)

print("=" * 60)
print("RESULTADOS DEL BENCHMARK")
print("=" * 60)
print(df_benchmark.to_string(index=False))

aciertos_total = (df_benchmark["Correcto"] == "Sí").sum()
print(f"
Precisión total: {aciertos_total}/{len(benchmark)} ({aciertos_total/len(benchmark)*100:.0f}%)")


Un **benchmark** es un conjunto estandarizado de pruebas que nos permite medir el rendimiento del modelo de forma sistemática y repetible. Este patrón es la base de los benchmarks industriales como MMLU, HellaSwag o HumanEval. Las ventajas de usar pandas para mostrar los resultados:
- Vista tabular clara y comparativa
- Fácil de exportar a CSV para análisis posteriores ()
- Se puede extender con más métricas (tiempo de respuesta, tokens usados, etc.)


**Celda 20: Resumen de lo aprendido**


In [ ]:
checklist = [
    "[x] Sección 1: Instalación y configuración de la API de OpenAI",
    "[x] Sección 2: Primera llamada a la API y función reutilizable preguntar()",
    "[x] Sección 3: Parámetros de generación (temperature, max_output_tokens, top_p)",
    "[x] Sección 4: Streaming de respuestas token a token",
    "[x] Sección 5: Ingeniería de prompts (system prompt, few-shot, chain-of-thought)",
    "[x] Sección 6: Salida estructurada en JSON y análisis con pandas",
    "[x] Sección 7: Fallos comunes — alucinaciones y sesgos",
    "[x] Sección 8: Evaluación básica — exact match, LLM-as-judge y benchmark"
]

print("=" * 60)
print("       RESUMEN DEL LABORATORIO")
print("=" * 60)
for item in checklist:
    print(item)
print("=" * 60)
print("Laboratorio completado con éxito!")


**Celda 21: Próximos pasos**


## Próximos pasos

Ahora que dominas los fundamentos de la API de OpenAI, estos son los próximos temas a explorar:

- **RAG (Retrieval-Augmented Generation)**: Conectar el LLM con bases de datos de conocimiento externas para reducir alucinaciones y actualizar el conocimiento del modelo sin reentrenamiento.
- **Fine-tuning**: Ajustar los pesos del modelo con datos propios para especializarlo en una tarea o dominio concreto, logrando mayor precisión que el prompting.
- **Agentes y herramientas (Agents & Tool Use)**: Crear sistemas donde el LLM puede usar herramientas externas (buscadores, calculadoras, APIs) para resolver tareas complejas de forma autónoma.
- **Embeddings y búsqueda semántica**: Usar representaciones vectoriales del texto para búsquedas por similaridad, clustering y sistemas de recomendación.
- **Evaluación avanzada**: Técnicas más robustas como BLEU, ROUGE, BERTScore, y frameworks como RAGAS o LangSmith para evaluar pipelines completos de LLMs en producción.
